# Elicitation - Pythia Model Family

160M, 410M, 1B, 2.8B, 12B     

## Setup

In [18]:
# Cell 0: Environment Detection
import sys
from pathlib import Path
import torch

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

Environment: Local


In [19]:
# Cell 1: Colab Only — Install pinned dependencies
# ⚠️ Restart runtime after running this cell, then skip to Cell 2
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6

In [20]:
# Cell 1a: Confirm Transformer Lens version
from importlib.metadata import version
print("TransformerLens version:", version("transformer-lens"))

TransformerLens version: 2.18.0


In [21]:
# Cell 2: Project Root & Path Setup
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_branch = "rearrange-results"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone -b {repo_branch} {repo_url} {PROJECT_ROOT}
else:
    # Local: notebook lives in notebooks/, project root is one level up
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Environment: Local
Project root: /Users/trishasalas/Repos/Research/tmlr


In [22]:
import importlib
importlib.invalidate_caches()
import src
print(src.__file__)

/Users/trishasalas/Repos/Research/tmlr/src/__init__.py


In [23]:
# Cell 3: Imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import src
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

# Device selection
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: mps


In [24]:
# Cell 5 - Model name variable
model_name = "pythia-160m"

In [25]:
# Cell 6: Load Model
model = HookedTransformer.from_pretrained(f"EleutherAI/{model_name}")

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

KeyboardInterrupt: 

In [ ]:
print(torch.cuda.is_available(), next(model.parameters()).device)

### Elicitation Battery

In [ ]:
# Elicitation Battery — Loads prompts from a YAML file, runs them through
# a model, and saves results to a per-model CSV.
import importlib
import yaml
import pandas as pd

prompt_files = [
    'control.yaml',
    'accessibility.yaml',
    'medical.yaml',
    'legal.yaml',
    'finance.yaml'
    ]

all_results = []

for prompts_file in prompt_files:
    domain = Path(prompts_file).stem
    prompts_path = PROJECT_ROOT / 'data' / prompts_file
    with open(prompts_path, 'r') as f:
        templates = yaml.safe_load(f)
    prompts = templates['prompts']
    print(f"\n--- Running {domain}: {len(prompts)} prompts ---")

    results = []
    for i, case in enumerate(prompts):
        print(f"\r  {i+1}/{len(prompts)}", end="")
        prompt = case['prompt']
        with torch.no_grad():
            full_output = model.generate(
                prompt,
                max_new_tokens=case['max_tokens'],
                temperature=0,
            )
        response = full_output[len(prompt):].strip()

        results.append({
            'domain': domain,          # <-- the missing key
            'prompt_id': case['prompt_id'],
            'concept': case['concept'],
            'prompt_type': case['prompt_type'],
            'template_type': case['template_type'],
            'prompt': prompt,
            'output': response,
            'max_tokens': case['max_tokens'],
            'model': model_name,
        })
    print()  # close the \r line so the next print doesn't collide

domain_df = pd.DataFrame(results)
output_path = PROJECT_ROOT / 'results' / 'pythia' / {model_name} / f'{model_name}-{domain}.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
domain_df.to_csv(output_path, index=False)
all_results.append(domain_df)

results_df = pd.concat(all_results, ignore_index=True)
print(f"\nSaved {len(results_df)} results to {output_path}")


--- Running control: 44 prompts ---
  1/44

  0%|          | 0/100 [00:00<?, ?it/s]

  2/44

  0%|          | 0/100 [00:00<?, ?it/s]

  3/44

...

  3/42

  0%|          | 0/100 [00:00<?, ?it/s]

  4/42

  0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
domain_df = pd.DataFrame(results)
output_path = PROJECT_ROOT / 'results' / 'gpt2' / {model_name} / f'{model_name}-{domain}.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
domain_df.to_csv(output_path, index=False)
all_results.append(domain_df)

results_df = pd.concat(all_results, ignore_index=True)
print(f"\nSaved {len(results_df)} results to {output_path}")

In [ ]:
output_dir = PROJECT_ROOT / 'results' / 'elicitation' / 'gpt2' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / f'{model_name}-elicitation.md', 'w') as f:
  f.write(f"# Model data captured during Elicitation Battery\n\n")
  f.write(f"- Model name: {model_name}\n")
  f.write(f"- Model dtype: {next(model.parameters()).dtype}\n")
  f.write(f"- Layers: {model.cfg.n_layers}\n")
  f.write(f"- Heads: {model.cfg.n_heads}\n")
  f.write(f"- Hidden size: {model.cfg.d_model}\n")
  f.write(f"- Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M\n")



print(f"Saved to {output_dir}")

In [ ]:
import os
os.chdir(PROJECT_ROOT)
!git config user.email "trisha@trishasalas.com"
!git config user.name "Trisha Salas"
!git add results/
!git commit -m "elicitation results: {model_name}"
!git push

### Delete Model & Clear Cache

In [ ]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")